# Genetic Algorithm Penjadwalan (Versi NumPy Optimized)
Notebook ini mempertahankan struktur fungsi asli tetapi mengganti struktur data menjadi NumPy agar lebih cepat.

Perubahan utama:
- Individu disimpan sebagai NumPy array
- Mapel dan guru dipisah menjadi matrix
- Operasi konflik menggunakan NumPy
- Mengurangi penggunaan dictionary di loop dalam


In [1]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict

## Parameter GA

In [2]:
POPULASI = 1000
VIOLATION_COST = 100
ITERATION = 1000
MUTATION_PROB = 0.7
TOURNAMENT_SIZE = 10

JUMLAH_KELAS = 27
SLOT_PER_KELAS = 36

## Load Dataset

In [3]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

## Precompute Mapping

In [4]:
mapel_ids = mapel_df['mapel_id'].to_numpy()
guru_ids = guru_df['guru_id'].to_numpy()
jam_per_minggu = dict(zip(mapel_df.mapel_id, mapel_df.jam_per_minggu))

wali_kelas = dict(zip(wali_kelas_df.guru_id, wali_kelas_df.kelas_id))

## Representasi Individu
Shape individu:
```
(kelas, slot, 2)
[:,:,0] = mapel
[:,:,1] = guru
```

In [5]:
def individuTrigger():
    individu = np.zeros((JUMLAH_KELAS, SLOT_PER_KELAS, 2), dtype=np.int16)

    for k in range(JUMLAH_KELAS):
        individu[k,:,0] = np.random.choice(mapel_ids, SLOT_PER_KELAS)
        individu[k,:,1] = np.random.choice(guru_ids, SLOT_PER_KELAS)

    return individu

## Populasi Awal

In [6]:
def populasiConstruct():
    return [individuTrigger() for _ in range(POPULASI)]

## Guru Bentrok (Vectorized)

In [7]:
def guruBentrok(individu):

    guru_matrix = individu[:,:,1]
    pelanggaran = 0

    for slot in range(SLOT_PER_KELAS):
        guru_slot = guru_matrix[:,slot]

        unique, counts = np.unique(guru_slot, return_counts=True)

        pelanggaran += np.sum(counts[counts > 1] - 1)

    return pelanggaran

## Durasi Guru Mengajar

In [8]:
def durasiGuru(individu):

    guru = individu[:,:,1].flatten()

    unique, counts = np.unique(guru, return_counts=True)

    pelanggaran = np.sum(np.maximum(0, counts - 40))

    return pelanggaran

## Fitness Function

In [9]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)
    pelanggaran += durasiGuru(individu)

    return pelanggaran * VIOLATION_COST

## Tournament Selection

In [10]:
def turnamen(populasi, fitnessPop):

    kandidat = random.sample(range(len(populasi)), TOURNAMENT_SIZE)

    terbaik = kandidat[0]

    for i in kandidat:
        if fitnessPop[i] < fitnessPop[terbaik]:
            terbaik = i

    return populasi[terbaik]

## Crossover

In [11]:
def crossover(parent1, parent2):

    child = parent1.copy()

    kelas = np.random.randint(0, JUMLAH_KELAS)

    child[kelas,:,:] = parent2[kelas,:,:]

    return child

## Mutation

In [12]:
def mutasi(individu):

    child = individu.copy()

    kelas = np.random.randint(0, JUMLAH_KELAS)

    np.random.shuffle(child[kelas])

    return child

## Genetic Algorithm

In [13]:
def geneticAlgorithm():

    populasi = populasiConstruct()

    bestIndividu = None
    bestFitness = float('inf')

    for gen in range(ITERATION):

        fitnessPop = []

        for individu in populasi:

            fitness = evaluasiIndividu(individu)

            fitnessPop.append(fitness)

            if fitness < bestFitness:
                bestFitness = fitness
                bestIndividu = individu

        print('Generasi:', gen, 'Best Fitness:', bestFitness)

        elitIndex = np.argsort(fitnessPop)[:2]

        populasiBaru = [populasi[i] for i in elitIndex]

        while len(populasiBaru) < POPULASI:

            parent1 = turnamen(populasi, fitnessPop)
            parent2 = turnamen(populasi, fitnessPop)

            child = crossover(parent1, parent2)

            if random.random() < MUTATION_PROB:
                child = mutasi(child)

            populasiBaru.append(child)

        populasi = populasiBaru

    return bestIndividu, bestFitness

In [14]:
hasil = geneticAlgorithm()

Generasi: 0 Best Fitness: 16000
Generasi: 1 Best Fitness: 15800
Generasi: 2 Best Fitness: 15000
Generasi: 3 Best Fitness: 14600
Generasi: 4 Best Fitness: 14300
Generasi: 5 Best Fitness: 13800
Generasi: 6 Best Fitness: 13600
Generasi: 7 Best Fitness: 13400
Generasi: 8 Best Fitness: 13100
Generasi: 9 Best Fitness: 12600
Generasi: 10 Best Fitness: 12400
Generasi: 11 Best Fitness: 12200
Generasi: 12 Best Fitness: 11900
Generasi: 13 Best Fitness: 11700
Generasi: 14 Best Fitness: 11200
Generasi: 15 Best Fitness: 11200
Generasi: 16 Best Fitness: 10900
Generasi: 17 Best Fitness: 10800
Generasi: 18 Best Fitness: 10400
Generasi: 19 Best Fitness: 10400
Generasi: 20 Best Fitness: 10400
Generasi: 21 Best Fitness: 10100
Generasi: 22 Best Fitness: 10100
Generasi: 23 Best Fitness: 9800
Generasi: 24 Best Fitness: 9800
Generasi: 25 Best Fitness: 9800
Generasi: 26 Best Fitness: 9600
Generasi: 27 Best Fitness: 9400
Generasi: 28 Best Fitness: 9400
Generasi: 29 Best Fitness: 9400
Generasi: 30 Best Fitness: 